# TealKit MCP Agent — Weather Forecast MCP — Qwen2.5-1.5B Training Pipeline

Fine-tunes **Qwen2.5-1.5B-Instruct** (`unsloth/Qwen2.5-1.5B-Instruct`) for Weather Forecast MCP tool-calling via Unsloth LoRA on Colab.

**Tool Set:** `get_current_weather`, `get_hourly_forecast`, `get_daily_forecast`, `geocode_weather_city`

**Key Features:**
1. **ChatML Format:** Uses `chat_template='chatml'` — clean `<|im_end|>` markers, compatible with Ollama.
2. **Configurable Context Window:** Set `MAX_SEQ_LENGTH` below (8K/16K/32K/64K).
3. **Loss-Masking:** Masks user prompt tokens so the model learns only assistant tool-call responses.
4. **Memory-Cleared Export:** Drops the trained model from GPU memory before GGUF export to prevent Colab OOM crashes.

## Cell 1 — Install Dependencies
> Installs the latest Unsloth and wipes the broken `llama.cpp` cache to prevent the 'conversion' module error.

In [ ]:
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers" trl peft accelerate bitsandbytes datasets huggingface_hub

import shutil
shutil.rmtree('/root/.unsloth', ignore_errors=True)

print('Install done. Restarting runtime...')
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

## Cell 2 — Config
> Set model, paths, and context window. Adjust `MAX_SEQ_LENGTH` based on your GPU memory.

In [ ]:
MODEL_NAME = 'unsloth/Qwen2.5-1.5B-Instruct'

# ── Context Window ──────────────────────────────────────────────────────────────
# Supported values for Qwen2.5-1.5B-Instruct:
#   8192   (8K)    — fits any GPU (T4/L4/H100)
#   16384  (16K)   — fits L4/H100, default
#   32768  (32K)   — native max, needs H100 or reduced batch size
#   65536  (64K)   — requires RoPE extension (YaRN), H100 recommended
# Note: larger contexts increase memory pressure. Reduce batch_size or
# enable gradient checkpointing if you hit OOM.
MAX_SEQ_LENGTH = 16384

DATA_DIR = '/content/drive/MyDrive/Tealkit/training/weatherforecast/mcp_out'
OUTPUT_DIR = '/content/drive/MyDrive/Tealkit/training/weatherforecast/mcp_adapters_qwen25_1p5b'
GGUF_DIR = '/content/drive/MyDrive/Tealkit/training/weatherforecast/mcp_fused_model_qwen25_1p5b'
MERGE_DIR = '/content/drive/MyDrive/Tealkit/training/weatherforecast/mcp_merged_model_qwen25_1p5b'
SYSTEM_PROMPT_FILE = '/content/drive/MyDrive/Tealkit/training/weatherforecast/weatherforecast_system_prompt.md'
PREFER_EMBEDDED_UPDATED_PROMPT = True

# Hugging Face repo for upload (Cell 10)
HF_REPO = 'lschaffer/qwen25-1p5b-weatherforecast'  # <-- Change to your HF username/repo

TRAIN_FILE = f'{DATA_DIR}/train_split.jsonl'
VALID_FILE = f'{DATA_DIR}/valid_split.jsonl'

print('Model                 :', MODEL_NAME)
print('MAX_SEQ_LENGTH        :', MAX_SEQ_LENGTH)
print('Train file            :', TRAIN_FILE)
print('Valid file            :', VALID_FILE)
print('Drive system prompt   :', SYSTEM_PROMPT_FILE)
print('Adapters out          :', OUTPUT_DIR)
print('GGUF out              :', GGUF_DIR)
print('HF repo               :', HF_REPO)


## Cell 3 — Mount Drive

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive', force_remount=True)

for _path, _label in [(TRAIN_FILE, 'train_split.jsonl'), (VALID_FILE, 'valid_split.jsonl'), (SYSTEM_PROMPT_FILE, 'weatherforecast system prompt')]:
    if not os.path.isfile(_path):
        print(f'MISSING {_label}: {_path} -> Upload to Drive first!')
    else:
        print(f'OK  {_label} found at {_path}')

## Cell 4 — Load model with Native bfloat16

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,   
    dtype=torch.bfloat16, 
)
print('Model loaded in native bfloat16 precision.')

## Cell 5 — Apply LoRA adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=3407,
)
model.print_trainable_parameters()

## Cell 6 — Load dataset + System Prompt

In [ ]:
from datasets import load_dataset
import json
import os
from unsloth.chat_templates import get_chat_template

# Force ChatML so markers perfectly align for response masking
tokenizer = get_chat_template(
    tokenizer,
    chat_template='chatml',
)

UPDATED_SYSTEM_PROMPT = '''# ROLE
You are a Weather Assistant with access to global weather data. You provide current conditions, hourly and daily forecasts, and can look up cities by name.

# THE 3 GOLDEN RULES
1. TOOL USE ONLY: Use the provided tools for all weather data. Do not simulate data or write code.
2. SILENT TOOL CALLS: When a tool is needed, output only the canonical tool call and nothing else.
3. FINAL ANSWERS ONLY AFTER DATA: If the user message already contains a tool result, answer directly from that result and do not call another tool.

# CANONICAL TOOL CALL FORMAT
- Use exactly: `tool_call: {"name":"<tool_name>","arguments":{...}}`
- Do not output XML tags or `{"tool_call":"..."}`.
- Do not use `parameters` instead of `arguments`.
- Do not add explanation before or after the tool call.

# CURRENT SCHEMA TOOLS
- `get_current_weather`: Get current weather conditions. Optional lat/lon.
- `get_hourly_forecast`: Get hourly forecast (24-168 hours). Params: hours, outputType, optional lat/lon.
- `get_daily_forecast`: Get daily forecast (1-16 days). Params: days, optional lat/lon.
- `geocode_weather_city`: Look up coordinates for a city name. Required: city.

# ORDER OF OPERATIONS
1. CITY LOOKUP: If the user mentions a city name without coordinates, use `geocode_weather_city` first.
2. CURRENT WEATHER: For "current weather", "weather now", use `get_current_weather`.
3. HOURLY FORECAST: For hourly forecast, use `get_hourly_forecast` with appropriate hours.
4. DAILY FORECAST: For daily/weekly forecast, use `get_daily_forecast` with appropriate days.

# PARAMETER RULES
- `latitude` and `longitude` must be floating-point numbers.
- Use `outputType: "pdf"` when the user explicitly asks for a PDF report.
- `hours` defaults to 24, `days` defaults to 7.

# FINAL ANSWER STYLE
- Keep final answers short, direct, factual.
- Present temperatures with units.
- Do not suggest extra work unless explicitly asked.'''

def is_modern_system_prompt(text):
    required_markers = [
        '# FINAL ANSWER STYLE',
        'tool_call: {"name":"<tool_name>","arguments":{...}}',
        'get_current_weather',
        'geocode_weather_city',
    ]
    return all(marker in text for marker in required_markers)

if os.path.isfile(SYSTEM_PROMPT_FILE):
    with open(SYSTEM_PROMPT_FILE, 'r', encoding='utf-8') as f:
        DRIVE_SYSTEM_PROMPT = f.read().strip()
    if PREFER_EMBEDDED_UPDATED_PROMPT and not is_modern_system_prompt(DRIVE_SYSTEM_PROMPT):
        SYSTEM_PROMPT = UPDATED_SYSTEM_PROMPT
        print('Drive system prompt is stale. Using embedded updated prompt instead.')
    else:
        SYSTEM_PROMPT = DRIVE_SYSTEM_PROMPT
        print('Loaded modern system prompt from Drive:', SYSTEM_PROMPT_FILE)
else:
    SYSTEM_PROMPT = UPDATED_SYSTEM_PROMPT
    print('Drive system prompt not found. Using embedded updated prompt.')

def format_example(examples):
    texts = []
    for msgs in examples['messages']:
        if not msgs or msgs[0].get('role') != 'system':
            msgs = [{'role': 'system', 'content': SYSTEM_PROMPT}] + list(msgs)
        texts.append(
            tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=False
            )
        )
    return {'text': texts}

dataset = load_dataset('json', data_files={'train': TRAIN_FILE, 'validation': VALID_FILE})
dataset = dataset.map(format_example, batched=True)

print('Train examples:', len(dataset['train']))
print('Valid examples:', len(dataset['validation']))

def _first_assistant_payload(messages):
    for message in messages:
        if not isinstance(message, dict) or message.get('role') != 'assistant':
            continue
        content = message.get('content', '')
        if not isinstance(content, str) or not content.startswith('tool_call: '):
            continue
        try:
            return json.loads(content[len('tool_call: '):])
        except json.JSONDecodeError:
            return None
    return None

def print_dataset_diagnostics(split_name, split_dataset):
    total = len(split_dataset)
    followup_rows = 0
    no_tool_rows = 0
    bad_no_tool_rows = 0

    for row in split_dataset:
        messages = row.get('messages', [])
        if not isinstance(messages, list):
            continue

        user_count = sum(1 for message in messages if isinstance(message, dict) and message.get('role') == 'user')
        assistant_count = sum(1 for message in messages if isinstance(message, dict) and message.get('role') == 'assistant')
        if user_count >= 2 and assistant_count >= 2:
            followup_rows += 1

        payload = _first_assistant_payload(messages)
        if not isinstance(payload, dict):
            continue

        if payload.get('name') == 'no_tool':
            no_tool_rows += 1

    print(f'{split_name}: total={total}, followup_rows={followup_rows}, no_tool_rows={no_tool_rows}')

print_dataset_diagnostics('train', dataset['train'])
print_dataset_diagnostics('validation', dataset['validation'])

def detect_chat_markers(sample_text):
    instruction_candidates = [
        '<|im_start|>user\\n',
        '<|im_start|>user<|im_sep|>',
        '<|user|>',
    ]
    response_candidates = [
        '<|im_start|>assistant\\n',
        '<|im_start|>assistant<|im_sep|>',
        '<|assistant|>',
    ]
    instruction_part = next((part for part in instruction_candidates if part in sample_text), None)
    response_part = next((part for part in response_candidates if part in sample_text), None)
    return instruction_part, response_part

_sample_text = dataset['train'][0]['text'] if len(dataset['train']) else ''
INSTRUCTION_PART, RESPONSE_PART = detect_chat_markers(_sample_text)
print('Instruction marker:', repr(INSTRUCTION_PART))
print('Response marker   :', repr(RESPONSE_PART))


## Cell 7 — Training Loop
> Adjust `per_device_train_batch_size` down if you increase `MAX_SEQ_LENGTH` (e.g., 2 for 32K, 1 for 64K).

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import DataCollatorForSeq2Seq
from unsloth.chat_templates import train_on_responses_only

# Reduce batch_size for larger contexts:
#   8192  -> batch_size=4
#   16384 -> batch_size=2
#   32768 -> batch_size=1 (with grad_accum=4)
#   65536 -> batch_size=1 (with grad_accum=8, may OOM on T4)
_batch_size = 4 if MAX_SEQ_LENGTH <= 8192 else (2 if MAX_SEQ_LENGTH <= 16384 else 1)
_grad_accum = 2 if MAX_SEQ_LENGTH <= 8192 else (4 if MAX_SEQ_LENGTH <= 32768 else 8)

_args = SFTConfig(
    per_device_train_batch_size=_batch_size,
    gradient_accumulation_steps=_grad_accum,
    warmup_steps=5,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=5,
    eval_strategy='epoch',
    save_strategy='no',
    optim='adamw_8bit',
    weight_decay=0.01,
    lr_scheduler_type='linear',
    seed=3407,
    output_dir='/content/outputs',
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
    packing=False,
    args=_args,
)

if INSTRUCTION_PART and RESPONSE_PART:
    try:
        masked_trainer = train_on_responses_only(
            trainer,
            instruction_part=INSTRUCTION_PART,
            response_part=RESPONSE_PART,
        )
        if len(masked_trainer.train_dataset) == 0:
            print('WARNING: Response masking removed every train sample. Falling back to full-sequence training.')
        else:
            trainer = masked_trainer
            print(f'Response masking active: {len(trainer.train_dataset)} train samples.')
    except Exception as e:
        print("WARNING: Response masking failed, using full sequence. Error:", e)
else:
    print('WARNING: Could not detect chat markers in formatted text. Using full-sequence training.')

trainer.train()
print('Training complete.')

## Cell 8 — Save adapters + robust GGUF export
> Saves adapters to Drive, tries native Unsloth GGUF export, and falls back to manual llama.cpp conversion.

In [ ]:
import gc, glob, os, shutil, subprocess, sys, torch
if 'OUTPUT_DIR' not in dir(): OUTPUT_DIR = '/content/drive/MyDrive/Tealkit/training/weatherforecast/mcp_adapters_qwen25_1p5b'
if 'GGUF_DIR' not in dir(): GGUF_DIR = '/content/drive/MyDrive/Tealkit/training/weatherforecast/mcp_fused_model_qwen25_1p5b'
if 'MERGE_DIR' not in dir(): MERGE_DIR = '/content/drive/MyDrive/Tealkit/training/weatherforecast/mcp_merged_model_qwen25_1p5b'
QUANT_METHOD = 'q4_k_m'
GGUF_BASENAME = f"{HF_REPO.split('/')[-1]}-unsloth"
GGUF_F16_PATH = os.path.join(GGUF_DIR, f'{GGUF_BASENAME}-F16.gguf')
GGUF_QUANT_PATH = os.path.join(GGUF_DIR, f'{GGUF_BASENAME}-{QUANT_METHOD.upper()}.gguf')
FINAL_GGUF_FILE = None
GGUF_FILENAME = None
SKIP_NATIVE_GGUF_EXPORT = 'phi-4' in MODEL_NAME.lower() or 'phi4' in MODEL_NAME.lower()

# 1. Save LoRA adapters to Drive
os.makedirs(OUTPUT_DIR, exist_ok=True)
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('Adapters successfully saved to Drive:', OUTPUT_DIR)

def run_checked(command, cwd=None, extra_env=None):
    env = os.environ.copy()
    for key in ('PYTHONPATH', 'PYTHONHOME', 'PYTHONSTARTUP', 'PYTHONUSERBASE'):
        env.pop(key, None)
    env['PYTHONNOUSERSITE'] = '1'
    if extra_env:
        for key, value in extra_env.items():
            if value is None:
                env.pop(key, None)
            else:
                env[key] = value
    print('>>', ' '.join(command))
    subprocess.run(command, cwd=cwd, env=env, check=True)

def clear_unsloth_llama_cpp_cache():
    unsloth_llama_cpp_dir = '/root/.unsloth/llama.cpp'
    if os.path.isdir(unsloth_llama_cpp_dir):
        shutil.rmtree(unsloth_llama_cpp_dir, ignore_errors=True)
        print('Cleared cached Unsloth llama.cpp checkout:', unsloth_llama_cpp_dir)

def refresh_tokenizer_files(merged_dir, base_model):
    try:
        from huggingface_hub import hf_hub_download
    except ImportError:
        print('WARNING: huggingface_hub not available; skipping tokenizer refresh.')
        return
    for filename in ('tokenizer_config.json', 'tokenizer.json', 'special_tokens_map.json'):
        try:
            source_path = hf_hub_download(repo_id=base_model, filename=filename)
            shutil.copy2(source_path, os.path.join(merged_dir, filename))
            print(f'Refreshed {filename} from {base_model}')
        except Exception as file_error:
            print(f'INFO: Could not refresh {filename} from {base_model}: {file_error}')
    tokenizer_model_path = os.path.join(merged_dir, 'tokenizer.model')
    if not os.path.exists(tokenizer_model_path):
        try:
            source_path = hf_hub_download(repo_id=base_model, filename='tokenizer.model')
            shutil.copy2(source_path, tokenizer_model_path)
        except Exception:
            pass

def find_quantize_binary(llama_cpp_dir):
    candidates = [
        os.path.join(llama_cpp_dir, 'build', 'bin', 'llama-quantize'),
        os.path.join(llama_cpp_dir, 'build', 'bin', 'quantize'),
        shutil.which('llama-quantize'),
        shutil.which('quantize'),
    ]
    for candidate in candidates:
        if candidate and os.path.isfile(candidate) and os.access(candidate, os.X_OK):
            return candidate
    return None

def ensure_quantize_binary(llama_cpp_dir):
    quantize_bin = find_quantize_binary(llama_cpp_dir)
    if quantize_bin:
        return quantize_bin
    try:
        run_checked(['cmake', '-S', '.', '-B', 'build', '-DBUILD_SHARED_LIBS=OFF', '-DGGML_CUDA=OFF'], cwd=llama_cpp_dir)
        run_checked(['cmake', '--build', 'build', '--config', 'Release', '-j2'], cwd=llama_cpp_dir)
    except Exception as build_error:
        print(f'WARNING: llama.cpp build failed, quantized GGUF may be unavailable. Error: {build_error}')
    return find_quantize_binary(llama_cpp_dir)

def install_llama_cpp_requirements(llama_cpp_dir, pydeps_dir):
    requirements_file = os.path.join(llama_cpp_dir, 'requirements.txt')
    if os.path.isdir(pydeps_dir):
        shutil.rmtree(pydeps_dir)
    os.makedirs(pydeps_dir, exist_ok=True)
    if os.path.isfile(requirements_file):
        try:
            run_checked([sys.executable, '-m', 'pip', 'install', '--upgrade', '--target', pydeps_dir, '-r', requirements_file])
        except Exception:
            run_checked([sys.executable, '-m', 'pip', 'install', '--upgrade', '--target', pydeps_dir, 'numpy', 'sentencepiece', 'protobuf', 'safetensors', 'transformers', 'huggingface_hub', 'tqdm'])

def ensure_llama_cpp_checkout(llama_cpp_dir, force_fresh=False):
    convert_script = os.path.join(llama_cpp_dir, 'convert_hf_to_gguf.py')
    conversion_init = os.path.join(llama_cpp_dir, 'conversion', '__init__.py')
    gguf_init = os.path.join(llama_cpp_dir, 'gguf-py', 'gguf', '__init__.py')
    if force_fresh and os.path.exists(llama_cpp_dir):
        shutil.rmtree(llama_cpp_dir, ignore_errors=True)
    needs_clone = force_fresh
    if os.path.exists(llama_cpp_dir) and not os.path.isdir(os.path.join(llama_cpp_dir, '.git')):
        shutil.rmtree(llama_cpp_dir, ignore_errors=True)
        needs_clone = True
    if not os.path.isdir(os.path.join(llama_cpp_dir, '.git')):
        needs_clone = True
    if not needs_clone:
        required_paths = [convert_script, conversion_init, gguf_init]
        if not all(os.path.isfile(path) for path in required_paths):
            print('WARNING: Existing llama.cpp checkout is missing new converter layout. Re-cloning...')
            shutil.rmtree(llama_cpp_dir, ignore_errors=True)
            needs_clone = True
    if needs_clone:
        if os.path.exists(llama_cpp_dir):
            shutil.rmtree(llama_cpp_dir, ignore_errors=True)
        run_checked(['git', 'clone', '--depth', '1', 'https://github.com/ggml-org/llama.cpp', llama_cpp_dir])
    else:
        try:
            run_checked(['git', 'fetch', '--depth', '1', 'origin'], cwd=llama_cpp_dir)
            run_checked(['git', 'reset', '--hard', 'FETCH_HEAD'], cwd=llama_cpp_dir)
        except Exception:
            shutil.rmtree(llama_cpp_dir, ignore_errors=True)
            run_checked(['git', 'clone', '--depth', '1', 'https://github.com/ggml-org/llama.cpp', llama_cpp_dir])
    missing_after_clone = [path for path in [convert_script, conversion_init, gguf_init] if not os.path.isfile(path)]
    if missing_after_clone:
        raise RuntimeError(f'Incomplete llama.cpp checkout, missing: {missing_after_clone}')
    return convert_script

def manual_llama_cpp_convert(merged_dir):
    clear_unsloth_llama_cpp_cache()
    refresh_tokenizer_files(merged_dir, MODEL_NAME)
    llama_cpp_dir = '/content/llama.cpp'
    pydeps_dir = '/content/llama_cpp_pydeps'
    convert_script = ensure_llama_cpp_checkout(llama_cpp_dir)
    gguf_py_dir = os.path.join(llama_cpp_dir, 'gguf-py')
    install_llama_cpp_requirements(llama_cpp_dir, pydeps_dir)
    isolated_pythonpath = os.pathsep.join([p for p in [pydeps_dir, gguf_py_dir, llama_cpp_dir] if p])
    quantize_bin = ensure_quantize_binary(llama_cpp_dir)
    extra_env = {'PYTHONPATH': isolated_pythonpath}
    convert_cmd = [sys.executable, '-S', convert_script, merged_dir, '--outfile', GGUF_F16_PATH, '--outtype', 'f16']
    try:
        run_checked(convert_cmd, cwd=llama_cpp_dir, extra_env=extra_env)
    except subprocess.CalledProcessError:
        print('WARNING: First manual GGUF conversion attempt failed. Retrying with fresh checkout...')
        refresh_tokenizer_files(merged_dir, MODEL_NAME)
        convert_script = ensure_llama_cpp_checkout(llama_cpp_dir, force_fresh=True)
        quantize_bin = ensure_quantize_binary(llama_cpp_dir)
        extra_env['PYTHONPATH'] = os.pathsep.join([p for p in [pydeps_dir, gguf_py_dir, llama_cpp_dir] if p])
        convert_cmd[2] = convert_script
        run_checked(convert_cmd, cwd=llama_cpp_dir, extra_env=extra_env)
    if quantize_bin:
        run_checked([quantize_bin, GGUF_F16_PATH, GGUF_QUANT_PATH, QUANT_METHOD.upper()])
        return GGUF_QUANT_PATH
    print('WARNING: llama.cpp quantizer not found. Keeping F16 GGUF only.')
    return GGUF_F16_PATH

# 2. Try native Unsloth GGUF export; fall back to manual llama.cpp conversion
if os.path.exists(GGUF_DIR):
    shutil.rmtree(GGUF_DIR)
os.makedirs(GGUF_DIR, exist_ok=True)

if not SKIP_NATIVE_GGUF_EXPORT:
    print(f'\\nExporting natively to {QUANT_METHOD} GGUF...')
    try:
        clear_unsloth_llama_cpp_cache()
        trainer.model.save_pretrained_gguf(GGUF_DIR, tokenizer, quantization_method=QUANT_METHOD)
        _gguf_files = sorted(glob.glob(f'{GGUF_DIR}/*.gguf'))
        if not _gguf_files:
            raise RuntimeError(f'No GGUF files were created in {GGUF_DIR}')
        FINAL_GGUF_FILE = _gguf_files[0]
        GGUF_FILENAME = os.path.basename(FINAL_GGUF_FILE)
        print('\\n✓ Native GGUF export complete!')
        for _p in _gguf_files:
            print(f'  {_p}  ({os.path.getsize(_p) / 1024**2:.0f} MB)')
    except Exception as _e:
        print(f'\\n⚠ Native GGUF export failed: {_e}')
        print('Falling back to manual llama.cpp conversion...')
else:
    print('\\nSkipping native Unsloth GGUF export (manual llama.cpp fallback only).')

if not FINAL_GGUF_FILE:
    print('Saving a merged 16-bit model and running manual llama.cpp conversion...')
    if os.path.exists(MERGE_DIR):
        shutil.rmtree(MERGE_DIR)
    trainer.model.save_pretrained_merged(MERGE_DIR, tokenizer, save_method='merged_16bit')
    print(f'Fallback merged model saved at: {MERGE_DIR}')
    FINAL_GGUF_FILE = manual_llama_cpp_convert(MERGE_DIR)
    GGUF_FILENAME = os.path.basename(FINAL_GGUF_FILE)
    print(f'✓ Manual GGUF export complete: {FINAL_GGUF_FILE}')

if not FINAL_GGUF_FILE or not os.path.isfile(FINAL_GGUF_FILE):
    raise RuntimeError('GGUF export failed: no .gguf file was created.')

if FINAL_GGUF_FILE.endswith('.ggu'):
    _fixed_final = FINAL_GGUF_FILE + 'f'
    os.replace(FINAL_GGUF_FILE, _fixed_final)
    FINAL_GGUF_FILE = _fixed_final

if not FINAL_GGUF_FILE.endswith('.gguf'):
    raise RuntimeError(f'Unexpected GGUF filename (expected .gguf): {FINAL_GGUF_FILE}')

GGUF_FILENAME = os.path.basename(FINAL_GGUF_FILE)
print(f'Final GGUF file: {FINAL_GGUF_FILE}')
torch.cuda.empty_cache()
gc.collect()


## Cell 9 — Generate Model Card (README.md)
> Creates a standard Hugging Face model card with an Ollama Modelfile template.

In [ ]:
import os

MODEL_CARD_PATH = f"{GGUF_DIR}/README.md"
MODEL_SHORT_NAME = MODEL_NAME.split('/')[-1]
GGUF_FILENAME = globals().get('GGUF_FILENAME') or f"{HF_REPO.split('/')[-1]}-unsloth-{QUANT_METHOD.upper()}.gguf"

model_card_content = f'''---
base_model: {MODEL_NAME}
library_name: unsloth
tags:
- mcp
- weather-forecast
- tool-calling
- gguf
---

# Weather Forecast MCP Agent - {MODEL_SHORT_NAME}

This model was fine-tuned for the Weather Forecast MCP tool set.

## Tools
- get_current_weather, get_hourly_forecast, get_daily_forecast, geocode_weather_city

## Files
- GGUF: {GGUF_FILENAME}
- Adapters: saved during notebook execution
'''

with open(MODEL_CARD_PATH, 'w', encoding='utf-8') as handle:
    handle.write(model_card_content)

print('Model card generated at:', MODEL_CARD_PATH)

UPLOAD_TO_HF = False  # set to True when you want to upload immediately

if UPLOAD_TO_HF:
    import getpass
    from huggingface_hub import HfApi, upload_folder
    try:
        hf_token = os.environ.get('HF_TOKEN') or getpass.getpass('HF token: ')
        api = HfApi(token=hf_token)
        api.create_repo(repo_id=HF_REPO, repo_type='model', exist_ok=True)
        upload_folder(
            repo_id=HF_REPO,
            folder_path=GGUF_DIR,
            repo_type='model',
            token=hf_token,
        )
        print('Uploaded GGUF folder to Hugging Face:', HF_REPO)
    except Exception as exc:
        print('HF upload skipped/failed:', exc)